# Решения: рекурсия, lambda, pipeline

**Для преподавателя.** Все задачи из `lesson.ipynb` и `homework.ipynb`.

In [ ]:
# Вложенный список (без dict)
NESTED_LIST = [1, [2, [3, 4]], 5]
# Простое дерево: (name, children)
CATEGORY_TREE = (
    'root',
    [
        ('electronics', [('phones', []), ('laptops', [])]),
        ('books', []),
    ],
)


## Урок. flatten

In [ ]:
def flatten(nested):
    flat = []
    for item in nested:
        if isinstance(item, list):
            flat.extend(flatten(item))
        else:
            flat.append(item)
    return flat


assert flatten(NESTED_LIST) == [1, 2, 3, 4, 5]

## Урок. walk_categories

In [ ]:
def walk_categories(node, depth=0):
    name, children = node
    print('  ' * depth + name)
    for child in children:
        walk_categories(child, depth + 1)


# walk_categories(CATEGORY_TREE)

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path('../..').resolve()))
from data.module_datasets import (
    APARTMENTS, EXAM_SCORES, PREDICTIONS, LABELS,
    NESTED_API_RESPONSE, CATEGORY_TREE, FEATURE_ROWS,
    MODEL_RUNS, FEATURE_POINTS,
    PRICE_INTERCEPT, PRICE_COEF_AREA,
)


## A1. extract_ids

In [ ]:
def extract_ids(nested):
    ids = []
    if isinstance(nested, dict):
        if 'id' in nested:
            ids.append(nested['id'])
        for value in nested.values():
            ids.extend(extract_ids(value))
    elif isinstance(nested, list):
        for item in nested:
            ids.extend(extract_ids(item))
    return ids


assert extract_ids(NESTED_API_RESPONSE) == [1, 2, 3]

## A3 / ДЗ. count_leaves

In [ ]:
def count_leaves(node):
    name, children = node
    if not children:
        return 1
    return sum(count_leaves(child) for child in children)


assert count_leaves(CATEGORY_TREE) == 3

## B1–B2. filter + leaderboard

In [ ]:
anomalies = list(filter(lambda s: s < 50 or s > 90, EXAM_SCORES))
assert set(anomalies) == {40, 48, 91, 95}
leaderboard = sorted(MODEL_RUNS, key=lambda r: (-r['f1'], r['id']))
assert [r['id'] for r in leaderboard] == [25, 30, 305, 101, 200]

## B3. farthest

In [ ]:
farthest = max(FEATURE_POINTS, key=lambda p: p[0] ** 2 + p[1] ** 2)
assert farthest == [55, 12]

## C1. apply_pipeline

In [ ]:
def apply_pipeline(data, steps):
    result = data
    for step in steps:
        result = step(result)
    return result


def extract_area(row):
    return row['area_sqm']


def scale_area(area, max_area=60):
    return area / max_area


def predict_from_scaled(scaled_area):
    return PRICE_INTERCEPT + PRICE_COEF_AREA * (scaled_area * 60)


apt_pipeline = [extract_area, scale_area, predict_from_scaled]
pred = apply_pipeline(FEATURE_ROWS[0], apt_pipeline)
assert abs(pred - (PRICE_INTERCEPT + PRICE_COEF_AREA * 28)) < 1e-9

## C2. text pipeline

In [ ]:
text_pipeline = [
    lambda s: s.strip().lower(),
    lambda s: s.split(),
    lambda tokens: [t for t in tokens if len(t) >= 4],
]
assert apply_pipeline('  Data science ML course  ', text_pipeline) == ['data', 'science', 'course']

## ДЗ. flatten_extra

In [ ]:
EXTRA = [0, [10, [20, 30]], 40]
assert flatten(EXTRA) == [0, 10, 20, 30, 40]